# Imports

In [2]:
from transformers import AutoTokenizer, BertForMultipleChoice
import pandas as pd
#from transformers import pipeline
from tqdm import tqdm
from datasets import load_dataset, load_from_disk

# restrict trainer to not use all cores
import os
#os.environ["OPENMP_NUM_THREADS"] = "3"

2025-12-04 15:58:39.596602: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-04 15:58:39.661331: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-04 15:58:41.005088: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# Data Loading
Load pre-saved dataset produced from the BERT-data-process notebook. Uses HuggingFace's dataset interface

In [3]:
dataset = load_from_disk("../data/bert-ds-labeled")
dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'inputs'],
        num_rows: 160028
    })
    validation: Dataset({
        features: ['labels', 'inputs'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['labels', 'inputs'],
        num_rows: 1000
    })
})

In [4]:
dataset['test'][0:4]

{'labels': [0, 2, 3, 3],
 'inputs': ['Facebook. I mentioned on Facebook that I was struggling for motivation to go for a run the other day, which has been translated by Tom’s great auntie as ‘Hayley can’t get out of bed’ and told to his grandma, who now thinks I’m a lazy, terrible person 🤣',
  "Amazon. BBC News - Amazon boss Jeff Bezos rejects claims company acted like a 'drug dealer' bbc.co.uk/news/av/busine…",
  'Microsoft. @Microsoft Why do I pay for WORD when it functions so poorly on my @SamsungUS Chromebook? 🙄',
  "CS-GO. CSGO matchmaking is so full of closet hacking, it's a truly awful game."]}

In [5]:
sentiments = ["Irrelevant","Positive","Neutral","Negative"]

# remove training set to speed up tokenization
dataset.pop("train")
dataset

DatasetDict({
    validation: Dataset({
        features: ['labels', 'inputs'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['labels', 'inputs'],
        num_rows: 1000
    })
})

In [6]:
dataset["test"][0:4]

{'labels': [0, 2, 3, 3],
 'inputs': ['Facebook. I mentioned on Facebook that I was struggling for motivation to go for a run the other day, which has been translated by Tom’s great auntie as ‘Hayley can’t get out of bed’ and told to his grandma, who now thinks I’m a lazy, terrible person 🤣',
  "Amazon. BBC News - Amazon boss Jeff Bezos rejects claims company acted like a 'drug dealer' bbc.co.uk/news/av/busine…",
  'Microsoft. @Microsoft Why do I pay for WORD when it functions so poorly on my @SamsungUS Chromebook? 🙄',
  "CS-GO. CSGO matchmaking is so full of closet hacking, it's a truly awful game."]}

## Tokenize

In [7]:
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')#"google-bert/bert-base-uncased")

In [8]:
# adapted from https://huggingface.co/docs/transformers/tasks/multiple_choice

count = 0
total = int(len(dataset["test"])/1000)

def preprocess(examples):
    global count, total
    count += 1
    tweet_text = [[content] * 4 for content in examples["inputs"]]
    #entities = examples["Entity"]

    tweet_sentiment = [
        [*sentiments] for label in examples["labels"]
    ]

    tweet_text = sum(tweet_text, [])
    tweet_sentiment = sum(tweet_sentiment, [])

    print(count,"/",total,"\t", len(tweet_text), len(tweet_sentiment))

    tokenized_examples = tokenizer(tweet_text, tweet_sentiment, truncation = True)
    return {k: [v[i : i+4] for i in range(0,len(v),4)] for k, v in tokenized_examples.items()}

In [9]:
#pbar = tqdm(total=len(dataset))
tokenized_data = dataset.map(preprocess, batched = True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

1 / 1 	 4000 4000


In [10]:
tokenized_data

DatasetDict({
    validation: Dataset({
        features: ['labels', 'inputs', 'input_ids', 'attention_mask'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['labels', 'inputs', 'input_ids', 'attention_mask'],
        num_rows: 1000
    })
})

# Prediction

## Load pretrained model
Load model trained by BERT-train notebook

In [11]:
from transformers import DataCollatorForMultipleChoice
collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

In [12]:
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer

model_name = "2neg6-checkpoint-90018"

model = AutoModelForMultipleChoice.from_pretrained('../models/%s'%(model_name))#"google-bert/bert-base-uncased")

## Predict
The prediction requires initializing a new trainer object and then using the trainer.predict() function

In [13]:
training_args = TrainingArguments(
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=None,
    eval_dataset=None,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=None,
)

print("Trainer Initialized")

y = trainer.predict(tokenized_data["test"])

/home/liam/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Trainer Initialized


In [ ]:
y_pred = [np.argmax(yi) for yi in y[0]]
y_act = y[1]
y[2]

# Metrics
Computation of metrics for the best DistilBERT model using the predictions made on the test set

In [ ]:
from sklearn.metrics import matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(y_act, y_pred, normalize="true"),display_labels=sentiments).plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix for DistilBERT")
plt.savefig("DistilBERT_CM.jpg",dpi=500)

In [ ]:
print("MCC:",matthews_corrcoef(y_act, y_pred))

In [ ]:
print(classification_report(y_act,y_pred,target_names=sentiments))

In [ ]:
confusion_matrix

# Plots
Plots of the loss and accuracy for the train and validaton sets across epochs using the output from the trainer 

In [ ]:
loss = '''1 	0.876700 	0.805823 	0.694667
2 	0.682000 	0.645796 	0.759333
3 	0.542600 	0.512632 	0.814000
4 	0.419200 	0.435329 	0.845333
5 	0.336600 	0.370833 	0.868000
6 	0.283500 	0.336668 	0.882667
7 	0.249300 	0.318494 	0.897333
8 	0.220100 	0.305934 	0.898000
9 	0.211300 	0.302102 	0.910000
10 	0.191000 	0.303720 	0.908000'''

train_loss = []
val_loss = []
epochs = []
accuracy = []

for epoch in loss.split("\n"):
    num, tl, vl, acc = epoch.split("\t")
    epochs.append(int(num))
    train_loss.append(float(tl))
    val_loss.append(float(vl))
    accuracy.append(float(acc))

In [ ]:

plt.plot(epochs,train_loss,label="train")
plt.plot(epochs,val_loss,label="validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.ylim(0,1)
plt.title("Training Loss for DistilBERT with $\\eta$=2e-6")
plt.legend()
plt.savefig("DB2e-6loss.jpg",dpi=500)

In [ ]:

plt.plot([9,9],[0,1],':r')
plt.plot(epochs,train_loss,label="train")
plt.plot(epochs,val_loss,label="validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.ylim(0,1)
plt.title("Training Loss for DistilBERT with $\\eta$=2e-6")
plt.legend()
plt.savefig("DB2e-6loss-line.jpg",dpi=500)

In [ ]:
val_loss

In [ ]:
plt.plot(epochs,accuracy)
plt.title("Accuracy per epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")